# Ch 1. 왜 모델이 필요한가

## 이 챕터에서 배우는 것

- **규칙(if-else) 로 풀 문제**와 **모델로 풀 문제**를 구분하는 직관
- AI Assistant 프로젝트를 시작하기 전 반드시 통과해야 하는 **3가지 판단 기준** (OpenAI 권장)
- 같은 문제를 룰 / 모델로 풀어보며 **어디서 룰이 깨지는지** 손으로 확인

원본 챕터: [왜 모델이 필요한가](https://desty.github.io/study-ai-assistant-engineering/part1/01-why-model/)

In [ ]:
# 환경 설치 (필수 라이브러리)
!pip install -q anthropic

In [ ]:
import os
from getpass import getpass
os.environ.setdefault('ANTHROPIC_API_KEY', getpass('ANTHROPIC_API_KEY: '))

## 1. 시작 — 작은 실험

아래 사용자 메시지 5개를 읽어보세요. 목표는 **"환불 요청인가 아닌가"** 를 판별하는 거예요.

In [ ]:
# 테스트 메시지들
messages = [
    "환불해주세요",
    "돈 돌려줘",
    "이거 생각했던 거랑 너무 달라요. 어떻게 해야 하나요?",
    "환불 정책이 어떻게 되나요?",
    "친구가 환불받았다고 하던데",
]

expected = [True, True, True, False, False]

for i, msg in enumerate(messages):
    print(f"{i+1}. [{expected[i]}] {msg}")

## 5. 최소 예제 — 룰이 깨지는 순간 직접 보기

In [ ]:
# 규칙 기반 접근
KEYWORDS = ["환불", "돌려", "취소"]

def is_refund_request_by_rule(msg: str) -> bool:
    return any(k in msg for k in KEYWORDS)

print("=== 규칙 기반 판별 ===")
results_rule = []
for m in messages:
    result = is_refund_request_by_rule(m)
    results_rule.append(result)
    print(f"[{result}]  {m}")

print(f"\n정확한 판별: {sum(r == e for r, e in zip(results_rule, expected))}/5")

## LLM으로 풀면

In [ ]:
from anthropic import Anthropic

client = Anthropic()

SYSTEM = """사용자 메시지가 '환불 요청'인지 판별하세요.
답: 'YES' 또는 'NO' 한 단어로만."""

print("=== LLM 기반 판별 ===")
results_llm = []
for m in messages:
    r = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=5,
        system=SYSTEM,
        messages=[{"role": "user", "content": m}],
    )
    response_text = r.content[0].text.strip().upper()
    is_refund = response_text == "YES"
    results_llm.append(is_refund)
    print(f"[{response_text}]  {m}")

print(f"\n정확한 판별: {sum(r == e for r, e in zip(results_llm, expected))}/5")

## 비교 분석

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "메시지": messages,
    "예상": expected,
    "규칙": results_rule,
    "LLM": results_llm,
    "규칙 정확": [r == e for r, e in zip(results_rule, expected)],
    "LLM 정확": [r == e for r, e in zip(results_llm, expected)],
})

print(comparison.to_string(index=False))
print(f"\n정확도 비교:")
print(f"  규칙: {comparison['규칙 정확'].sum()}/{len(expected)}")
print(f"  LLM:  {comparison['LLM 정확'].sum()}/{len(expected)}")

## 다음 장

모델을 쓰기로 판단했다면, 그 "모델"이 어떻게 작동하는지 최소한은 알아야 합니다.

[Ch 2. LLM이란 무엇인가](02-what-is-llm.ipynb) 로 이동